# LyricInsight 감정 분석 모델 학습 (Colab용)

이 노트북은 LyricInsight 프로젝트의 한국어 가사 감정 분석 모델(`klue/roberta-base`)을 학습하기 위해 작성되었습니다.

## 1. 사전 준비
이 노트북을 실행하기 전에 왼쪽 **파일(Files)** 탭에 다음 파일들을 업로드해주세요:
1. `train.jsonl`
2. `val.jsonl`
3. `labels.json`

위 파일들은 로컬 프로젝트의 `d:\LyricInsight\data\processed_kpoem\` 경로에 있습니다.

In [ ]:
# 2. 필요 라이브러리 설치
!pip install transformers datasets scikit-learn accelerate torch

In [ ]:
import json
from pathlib import Path
import numpy as np
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import torch

# GPU 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 3. 데이터 로드 및 설정

# Colab 환경에서는 현재 디렉토리(/content/)를 사용
DATA_DIR = Path(".")

# 데이터 파일 경로
TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH = DATA_DIR / "val.jsonl"
LABELS_PATH = DATA_DIR / "labels.json"

# 출력 모델 경로
OUT_MODEL_DIR = Path("./emotion_v2")

def load_jsonl(path: Path):
    rows = []
    if not path.exists():
        raise FileNotFoundError(f"{path} 파일을 찾을 수 없습니다. Colab의 파일 탭에 업로드했는지 확인해주세요.")
        
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

if not LABELS_PATH.exists():
    print("Warning: labels.json not found. Please upload it.")
else:
    label_names = json.loads(LABELS_PATH.read_text(encoding="utf-8"))
    print("Labels:", label_names)
    print("Num labels:", len(label_names))

In [ ]:
# 4. 데이터셋 준비

MODEL_NAME = "klue/roberta-base"

train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

def cast_labels(batch):
    batch["labels"] = np.array(batch["labels"], dtype=np.float32)
    return batch

train_ds = train_ds.map(cast_labels)
val_ds = val_ds.map(cast_labels)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# 5. 모델 학습 설정 및 시작

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)

    # threshold
    thr = 0.3
    preds = (probs >= thr).astype(int)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "f1_micro": micro,
        "f1_macro": macro,
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    problem_type="multi_label_classification",
)

args = TrainingArguments(
    output_dir=str(OUT_MODEL_DIR),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16, # Colab GPU라 배치 사이즈 좀 늘림
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    save_total_limit=2,
    fp16=True, # GPU 가속 활용
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

In [ ]:
# 6. 모델 저장

OUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUT_MODEL_DIR))
tokenizer.save_pretrained(str(OUT_MODEL_DIR))

# labels.json 복사
(OUT_MODEL_DIR / "labels.json").write_text(
    json.dumps(label_names, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Model saved to", OUT_MODEL_DIR)

In [ ]:
# 7. 모델 다운로드 압축
!zip -r emotion_v2.zip emotion_v2

In [ ]:
# 8. 다운로드 (브라우저 다운로드 창이 뜨지 않으면 왼쪽 파일 탭에서 직접 다운로드 가능)
from google.colab import files
files.download('emotion_v2.zip')